# Complementary (Grounded) Concentric Transmon
## Full Design, Simulation and Quantum Analysis — Qiskit-Metal + pyEPR

### Geometry (v2 — explicit metallic plane)

```
        ┌─────────────────────────────────┐
        │           W x W                 │
        │       metallic plane            │
        │                                 │
        │        ╭──────────╮  ←Wslot     │
        │      ╭─╯  ○ disk  ╰─╮           │
        │  ──JJ─  Rdisk   JJ──            │
        │      ╰─╮          ╭─╯           │
        │        ╰──────────╯             │
        └─────────────────────────────────┘
```

| Parameter | Description | Design knob |
|---|---|---|
| `W` | Side of outer metallic square | Isolation / boundary condition |
| `Rdisk` | Radius of central metallic disk | Primary Ec knob (↑Rdisk → ↓Ec) |
| `Wslot` | Width of circular slot (gap) | Secondary Ec knob (↑Wslot → ↓Ec) |
| `jj_width` | Tangential width of each JJ bridge | JJ area / Ic spread |
| `Lj` | Josephson inductance (HFSS variable) | Ej → f_01 |

### Design targets
| Parameter | Target |
|---|---|
| f_01 | 3 – 4.8 GHz |
| Ec | ~140 MHz |
| Ej/Ec | > 50 |

### Reference: rfphotoniqlabs/qiskit-metal
- `tutorials/4 Analysis/A. Core/4.02 Eigenmode and EPR.ipynb`


In [1]:
%reload_ext autoreload
%autoreload 2

import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import brentq

import qiskit_metal as metal
from qiskit_metal import designs, draw
from qiskit_metal import MetalGUI, Dict
from qiskit_metal.toolbox_python.attr_dict import Dict
from qiskit_metal.qlibrary.core import QComponent

print(f"Qiskit-Metal version: {metal.__version__}")


Qiskit-Metal version: 0.5.3.post1


07:48AM 01s WARNING [_qt_message_handler]: WARNING: "Unable to open monitor interface to \\\\.\\DISPLAY433:" "Unknown error 0xe0000225." (No context available from Qt)
Python Traceback (most recent call last):
  File "C:\Users\Kobi\anaconda3\envs\quantum_metal_env\Lib\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "C:\Users\Kobi\anaconda3\envs\quantum_metal_env\Lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "C:\Users\Kobi\anaconda3\envs\quantum_metal_env\Lib\site-packages\ipykernel\kernelapp.py", line 758, in start
    self.io_loop.start()
  File "C:\Users\Kobi\anaconda3\envs\quantum_metal_env\Lib\site-packages\tornado\platform\asyncio.py", line 211, in start
    self.asyncio_loop.run_forever()
  File "C:\Users\Kobi\anaconda3\envs\quantum_metal_env\Lib\asyncio\base_events.py", line 608, in run_forever
    self._run_once()
  File "C:\Users\Kobi\anaconda3\envs\quantum_metal_env\Lib\

---
## Step 1 — Load `CircmonG` (single-JJ grounded circular transmon)

The `CircmonG` component is imported from the user-components library.

**Geometry**
- `pad`      : filled circle of radius `pad_r` — the qubit island
- `pocket`   : annular ring of width `pad2pk_gap`, subtracted from the ground plane
- `rect_jj1` : **single** Josephson Junction at azimuthal angle `jj_angle`

| Option | Default | Description |
|---|---|---|
| `pad_r` | `'100um'` | Qubit island radius |
| `pad2pk_gap` | `'100um'` | Circular slot width |
| `jj_width` | `'20um'` | JJ bridge width |
| `jj_angle` | `0` | JJ angle in degrees (0 = east) |
| `jj_in_pad` | `'2um'` | JJ overlap into pad edge |
| `jj_in_pk` | `'2um'` | JJ overlap into pocket edge |

**HFSS junction object names** (auto-generated by Qiskit-Metal renderer):
```
rect : JJ_rect_Lj_{component_name}_rect_jj1
line : JJ_Lj_{component_name}_rect_jj1_
```


In [2]:
from qiskit_metal.qlibrary.user_components.circmong_doubleJJ import CircmonG
print("CircmonG imported successfully.")
print()
print("Junction keys in HFSS: 'rect_jj1', 'rect_jj2'")
print("For component name 'TC1':")
print("  rect_jj1 rect : JJ_rect_Lj_TC1_rect_jj1")
print("  rect_jj1 line : JJ_Lj_TC1_rect_jj1_")
print("  rect_jj2 rect : JJ_rect_Lj_TC1_rect_jj2")
print("  rect_jj2 line : JJ_Lj_TC1_rect_jj2_")


CircmonG imported successfully.

Junction keys in HFSS: 'rect_jj1', 'rect_jj2'
For component name 'TC1':
  rect_jj1 rect : JJ_rect_Lj_TC1_rect_jj1
  rect_jj1 line : JJ_Lj_TC1_rect_jj1_
  rect_jj2 rect : JJ_rect_Lj_TC1_rect_jj2
  rect_jj2 line : JJ_Lj_TC1_rect_jj2_


---
## Step 2 — Build the Design and Display in MetalGUI


In [3]:
# ── Create design ─────────────────────────────────────────────────────────
# Set chip size to match W (the metallic plane side).
# A small border is added so the HFSS airbox does not clip the component.
W_um = 800    # must match the W option below
H_um = 280

design = designs.DesignPlanar({}, True)
design.chips.main.size['size_x'] = f'{W_um}um'
design.chips.main.size['size_y'] = f'{W_um}um'
design.chips.main.size['size_z'] = f'{-H_um}um'
design.chips.main.size['center_x'] = '0um'
design.chips.main.size['center_y'] = '0um'

gui = MetalGUI(design)
print("Design created. MetalGUI opened.")


Design created. MetalGUI opened.


In [4]:
from qiskit_metal.qlibrary.user_components.circmong_doubleJJ import CircmonG

circmon1 = CircmonG(design, 'TC1', options=dict(
    pos_x           = '0mm',
    pos_y           = '0mm',
    pad_r           = '200um',   # qubit island radius — primary Ec knob
    pad2pk_gap      = '100um',   # slot width          — secondary Ec knob
    jj_width        = '20um',
    jj_angle        = 0,         # 0 deg = east (+x)
    jj_offset_angle = 180,       # second JJ at 180 deg offset (west)
    jj_in_pad       = '2um',
    jj_in_pk        = '2um',
))


In [5]:
design.overwrite_enabled = True
gui.rebuild()
gui.autoscale()

print("CircmonG built.")
print()
print(f"  pad_r           : {circmon1.options.pad_r}")
print(f"  pad2pk_gap      : {circmon1.options.pad2pk_gap}")
print(f"  jj_width        : {circmon1.options.jj_width}")
print(f"  jj_angle        : {circmon1.options.jj_angle} deg")
print(f"  jj_offset_angle : {circmon1.options.jj_offset_angle} deg")
print()
print("Junction table:")
print(design.qgeometry.tables['junction'][
    ['component','name','width','hfss_inductance','hfss_capacitance']])


CircmonG built.

  pad_r           : 200um
  pad2pk_gap      : 100um
  jj_width        : 20um
  jj_angle        : 0 deg
  jj_offset_angle : 180 deg

Junction table:
  component      name  width hfss_inductance hfss_capacitance
0         1  rect_jj1   0.02             Lj1               Cj
1         1  rect_jj2   0.02             Lj2               Cj


In [6]:
# ── Verify geometry using matplotlib renderer ─────────────────────────────
fig, ax = plt.subplots(1, 1, figsize=(6, 6))
design.renderers.mpl.render(ax=ax)
ax.set_title('CircmonG — Single JJ Layout', fontsize=12)
ax.set_xlabel('x (mm)')
ax.set_ylabel('y (mm)')
plt.tight_layout()
plt.savefig('circmong_layout.png', dpi=150, bbox_inches='tight')
plt.show()
print("Layout saved: circmong_layout.png")


TypeError: 'Dict' object is not callable

In [7]:
# ── Analytical preview: f_01 and Ej/Ec vs Lj (fixed Ec = 140 MHz) ────────
# Single JJ: Ej_eff = Ej_single  (no parallel combination)
h_planck = 6.626e-34
Phi0     = 2.068e-15

def Lj_to_Ej_GHz(Lj_nH):
    """Single-JJ Ej in GHz from Lj in nH."""
    return (Phi0/(2*np.pi))**2 / (Lj_nH*1e-9) / (h_planck*1e9)

def f01_approx(Ej_GHz, Ec_GHz):
    """f_01 = sqrt(8*Ej*Ec) - Ec  [transmon approximation]."""
    return np.sqrt(8*Ej_GHz*Ec_GHz) - Ec_GHz

Ec = 0.140   # GHz — target Ec
print(f"Analytical preview  |  Ec = {Ec*1e3:.0f} MHz  |  single JJ")
print(f"{'Lj (nH)':>10} {'Ej (GHz)':>12} {'Ej/Ec':>8} {'f_01 (GHz)':>12}")
print("-" * 48)
for Lj_nH in [8, 9, 10, 11, 12, 13, 14, 15, 20, 25, 30]:
    Ej   = Lj_to_Ej_GHz(Lj_nH)   # single JJ — no x2
    ratio = Ej / Ec
    f01   = f01_approx(Ej, Ec)
    tag   = "  <-- target" if 3.0 <= f01 <= 4.8 else ""
    print(f"{Lj_nH:>10.0f} {Ej:>12.3f} {ratio:>8.0f} {f01:>12.4f}{tag}")


Analytical preview  |  Ec = 140 MHz  |  single JJ
   Lj (nH)     Ej (GHz)    Ej/Ec   f_01 (GHz)
------------------------------------------------
         8       20.436      146       4.6442  <-- target
         9       18.166      130       4.3706  <-- target
        10       16.349      117       4.1391  <-- target
        11       14.863      106       3.9400  <-- target
        12       13.624       97       3.7663  <-- target
        13       12.576       90       3.6130  <-- target
        14       11.678       83       3.4765  <-- target
        15       10.899       78       3.3539  <-- target
        20        8.174       58       2.8858
        25        6.540       47       2.5663
        30        5.450       39       2.3305


---
## Step 3 — Render to HFSS and Configure Josephson Junctions

### What `sim.run()` does in HFSS
1. Creates a new eigenmode design named `Qbit_hfss`
2. Exports `metal_plane` (W×W with hole) → HFSS PEC sheet on substrate top face
3. Exports `disk` → HFSS PEC sheet (the qubit island)
4. Creates silicon substrate box (εr = 11.7) sized to the chip
5. Exports `jj_east` / `jj_west` → HFSS lumped RLC elements
6. Assigns `Lj` and `Cj` as HFSS design variables on each lumped element

### JJ HFSS object naming convention
Qiskit-Metal's HFSS renderer auto-generates names from the component name and
the key passed to `add_qgeometry('junction', {key: geom})`:
```
rect : JJ_rect_Lj_{component_name}_{key}
line : JJ_Lj_{component_name}_{key}_
```
For component `CCT1` with keys `jj_east` and `jj_west`:
```
jj_east rect : JJ_rect_Lj_CCT1_jj_east
jj_east line : JJ_Lj_CCT1_jj_east_
jj_west rect : JJ_rect_Lj_CCT1_jj_west
jj_west line : JJ_Lj_CCT1_jj_west_
```


In [8]:
from qiskit_metal.analyses.quantization import EPRanalysis

# ── Create EPRanalysis object ─────────────────────────────────────────────
eig_qb = EPRanalysis(design, "hfss")

# ── Simulation setup ──────────────────────────────────────────────────────
eig_qb.sim.setup.name          = 'Qbit_Setup'
eig_qb.sim.setup.n_modes       = 1
eig_qb.sim.setup.max_passes    = 15
eig_qb.sim.setup.max_delta_f   = 0.1    # convergence: 0.1% change in freq
eig_qb.sim.setup.min_freq_ghz  = 1.0
eig_qb.sim.setup.min_converged = 2

# ── Junction variable (dual JJ) ───────────────────────────────────────────
# Each JJ carries inductance Lj; two JJs in parallel -> Lj_eff = Lj/2,
# Ej_eff = 2*(Phi0/2pi)^2/Lj  (double the single-JJ value for same Lj).
# To hit the same f_01 target as a single-JJ design, use Lj ~ 2 * Lj_single.
# For f_01 ~ 3.5 GHz at Ec ~ 140 MHz: Lj_single ~ 13.5 nH -> Lj ~ 27 nH.
eig_qb.sim.setup.vars = Dict(
    Lj1 = '27 nH',   # inductance of EACH JJ — adjust to tune f_01
    Lj2 = '27 nH',   # inductance of EACH JJ — adjust to tune f_01
    Cj = '2 fF',    # shunt capacitance
)

# ── HFSS box buffer ───────────────────────────────────────────────────────
eig_qb.sim.renderer.options['x_buffer_width_mm'] = 0.5
eig_qb.sim.renderer.options['y_buffer_width_mm'] = 0.5

print("EPRanalysis setup:")
eig_qb.sim.setup


EPRanalysis setup:


{'name': 'Qbit_Setup',
 'reuse_selected_design': True,
 'reuse_setup': True,
 'min_freq_ghz': 1.0,
 'n_modes': 1,
 'max_delta_f': 0.1,
 'max_passes': 15,
 'min_passes': 1,
 'min_converged': 2,
 'pct_refinement': 30,
 'basis_order': 1,
 'vars': {'Lj1': '27 nH', 'Lj2': '27 nH', 'Cj': '2 fF'}}

In [10]:
# ── Render design to HFSS ─────────────────────────────────────────────────
# Prerequisite: Ansys HFSS must be running with a valid licence.

eig_qb.sim.run(
    name='Qbit_hfss',
    components=['TC1'],
    open_terminations=[],
    box_plus_buffer=True,
)

eig_qb.sim.print_run_args()
print()
print("HFSS rendering complete.")
print()

# ── Confirm JJ object names that pyEPR will look for ─────────────────────
comp = 'TC1'
print(f"Expected HFSS JJ object names for component '{comp}':")
print(f"  rect_jj1 rect : JJ_rect_Lj_{comp}_rect_jj1")
print(f"  rect_jj1 line : JJ_Lj_{comp}_rect_jj1_")
print(f"  rect_jj2 rect : JJ_rect_Lj_{comp}_rect_jj2")
print(f"  rect_jj2 line : JJ_Lj_{comp}_rect_jj2_")


INFO 11:57PM [connect_design]: 	Opened active design
	Design:    Qbit_hfss_hfss [Solution type: Eigenmode]
INFO 11:57PM [get_setup]: 	Opened setup `Qbit_Setup`  (<class 'pyEPR.ansys.HfssEMSetup'>)
INFO 11:57PM [analyze]: Analyzing setup Qbit_Setup
12:00AM 27s INFO [get_f_convergence]: Saved convergences to C:\Users\Kobi\Documents\quantum_metal_env\quantum-metal\Kobis_projects\Tunable_transmon\hfss_eig_f_convergence.csv


This analysis object run with the following kwargs:
{'name': 'Qbit_hfss', 'components': ['TC1'], 'open_terminations': [], 'port_list': None, 'jj_to_port': None, 'ignored_jjs': None, 'box_plus_buffer': True}


HFSS rendering complete.

Expected HFSS JJ object names for component 'TC1':
  rect_jj1 rect : JJ_rect_Lj_TC1_rect_jj1
  rect_jj1 line : JJ_Lj_TC1_rect_jj1_
  rect_jj2 rect : JJ_rect_Lj_TC1_rect_jj2
  rect_jj2 line : JJ_Lj_TC1_rect_jj2_


---
## Step 4 — Run Finite-Element Eigenmode Analysis in HFSS

HFSS replaces each Josephson Junction with its linearised inductance Lj and
solves Maxwell's eigenvalue problem for the closed cavity.  The resulting
eigenfrequency is the **bare (linearised) qubit frequency** — slightly higher
than the true dressed frequency because the cosine nonlinearity is not yet
included (that is corrected in Step 9 by pyEPR quantum analysis).

Convergence criterion: < 0.1% change in eigenfrequency between consecutive
adaptive mesh refinement passes.


In [11]:
# ── The HFSS solve was triggered by sim.run() in Step 3. ──────────────────
# Use this cell to check convergence and re-run if needed.

# Re-run after changing setup parameters:
# eig_qb.sim.renderer.analyze_setup(eig_qb.sim.setup.name)

# ── Plot convergence ──────────────────────────────────────────────────────
'''
eig_qb.sim.plot_convergences()
plt.suptitle(
    "HFSS Eigenmode Convergence — Grounded Concentric Transmon",
    fontsize=11, y=1.02
)
plt.tight_layout()
plt.savefig('convergence.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Extract eigenfrequencies ──────────────────────────────────────────────
freqs = eig_qb.get_frequencies()
print("Eigenmode frequencies (linearised with Lj):")
print(freqs)
print()
print("Convergence per pass:")
print(eig_qb.sim.convergence_f)'''

import pandas as pd

conv_t = eig_qb.sim.convergence_t
conv_f = eig_qb.sim.convergence_f

if isinstance(conv_t, pd.DataFrame) and isinstance(conv_f, (pd.DataFrame, pd.Series)):
    eig_qb.sim.plot_convergences(convergence_t=conv_t, convergence_f=conv_f)
    plt.suptitle(
        "HFSS Eigenmode Convergence — Grounded Concentric Transmon",
        fontsize=11, y=1.02
    )
    plt.tight_layout()
    plt.savefig('convergence.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print("Convergence data not yet populated as pandas — skipping plot.")
    print(f"  convergence_t : {type(conv_t)}")
    print(f"  convergence_f : {type(conv_f)}")

# ── Extract eigenfrequencies ──────────────────────────────────────────────
freqs = eig_qb.get_frequencies()
print("Eigenmode frequencies (linearised with Lj):")
print(freqs)
print()
print("Convergence per pass:")
print(conv_f)

Design "Qbit_hfss_hfss" info:
	# eigenmodes    1
	# variations    1
Design "Qbit_hfss_hfss" info:
	# eigenmodes    1
	# variations    1
Eigenmode frequencies (linearised with Lj):
                Freq. (GHz)  Quality Factor
variation mode                             
0         0        3.779645             inf

Convergence per pass:
         re(Mode(1)) [g]
Pass []                 
1               2.689629
2               3.520542
3               3.709117
4               3.739887
5               3.755719
6               3.764103
7               3.770097
8               3.774588
9               3.777485
10              3.779645
11                   NaN
12                   NaN
13                   NaN
14                   NaN
15                   NaN


---
## Step 5 — Plot Electric and Magnetic Field Distributions

Expected field pattern for the **grounded concentric transmon**:

| Field | Expected distribution |
|---|---|
| **E-field** | Radially concentrated across the circular slot (Wslot gap). Strong radial E in the slot, falls off quickly inside the disk and in the metallic plane. |
| **H-field** | Circulates around the disk; peaks at the JJ bridges (east/west) where current loops close through the JJ inductance. |

A large fraction of E-field inside the substrate is expected (~90%),
consistent with typical planar transmon designs on silicon.


In [12]:
# ── E-field magnitude on chip surface ────────────────────────────────────
print("Plotting E-field (eigenmode 1)...")
eig_qb.sim.plot_fields('main')
eig_qb.sim.save_screenshot('efield_main.png')
print("  Saved: efield_main.png")

# ── H-field ──────────────────────────────────────────────────────────────
print("Plotting H-field (eigenmode 1)...")
eig_qb.sim.plot_fields('main', field_type='H')
eig_qb.sim.save_screenshot('hfield_main.png')
print("  Saved: hfield_main.png")

eig_qb.sim.clear_fields()

print()
print("Interpretation checklist:")
print("  [ ] E-field peaks inside the circular slot (Wslot gap region)")
print("  [ ] E-field drops sharply outside slot and inside disk")
print("  [ ] H-field circulates around disk; peaks at +/-x JJ positions")
print("  [ ] No significant field at chip boundary (W x W square edge)")
print()
print("If E-field leaks to the chip boundary, increase W to reduce")
print("boundary coupling and improve accuracy.")


INFO 12:00AM [get_setup]: 	Opened setup `Qbit_Setup`  (<class 'pyEPR.ansys.HfssEMSetup'>)


Plotting E-field (eigenmode 1)...


TypeError: QSimulation.save_screenshot() takes 1 positional argument but 2 were given

---
## Step 6 — EPR Analysis on the Qubit Mode

pyEPR integrates the stored electric energy over each JJ rectangle to
compute the inductive Energy Participation Ratio (EPR):

```
p_mj = (inductive energy in JJ j for mode m) / (total electric energy)
```

For the grounded concentric transmon with two symmetric JJs:
- `p_total = p_east + p_west`
- Ideally `p_total → 1` (all inductive energy in the JJs)
- Typical values: 0.85 – 0.98 for well-designed transmons

The capacitive EPR `p_cap` (small for transmon) and sign `s_mj` are also reported.


In [13]:
# ── Configure both junctions for EPR ─────────────────────────────────────
# Component name must match what was used in CircmonG(design, 'TC1', ...)
comp = 'TC1'

# Remove any default placeholder
if 'jj' in eig_qb.setup.junctions:
    del eig_qb.setup.junctions['jj']

# Register JJ1  (key used in make_jj() was 'rect_jj1')
# HFSS object names follow the convention:
#   rect : JJ_rect_Lj_{comp}_{key}
#   line : JJ_Lj_{comp}_{key}_
eig_qb.setup.junctions['rect_jj1'] = Dict(
    rect        = f'JJ_rect_Lj_{comp}_rect_jj1',
    line        = f'JJ_Lj_{comp}_rect_jj1_',
    Lj_variable = 'Lj1',
    Cj_variable = 'Cj',
)

# Register JJ2  (key used in make_jj() was 'rect_jj2')
eig_qb.setup.junctions['rect_jj2'] = Dict(
    rect        = f'JJ_rect_Lj_{comp}_rect_jj2',
    line        = f'JJ_Lj_{comp}_rect_jj2_',
    Lj_variable = 'Lj2',
    Cj_variable = 'Cj',
)

# ── Substrate dielectric for loss participation ───────────────────────────
chip_name = design.chips.main['_short_name']  # usually 'main'
substrate_obj = 'main'
eig_qb.setup.dissipatives.dielectrics_bulk = [substrate_obj]

print("EPR junction configuration (dual JJ):")
print(f"  JJ1 rect : JJ_rect_Lj_{comp}_rect_jj1")
print(f"  JJ1 line : JJ_Lj_{comp}_rect_jj1_")
print(f"  JJ2 rect : JJ_rect_Lj_{comp}_rect_jj2")
print(f"  JJ2 line : JJ_Lj_{comp}_rect_jj2_")
print(f"  Substrate obj : {substrate_obj}")
print()
print(eig_qb.setup)


EPR junction configuration (dual JJ):
  JJ1 rect : JJ_rect_Lj_TC1_rect_jj1
  JJ1 line : JJ_Lj_TC1_rect_jj1_
  JJ2 rect : JJ_rect_Lj_TC1_rect_jj2
  JJ2 line : JJ_Lj_TC1_rect_jj2_
  Substrate obj : main

{'junctions': {'rect_jj1': {'rect': 'JJ_rect_Lj_TC1_rect_jj1', 'line': 'JJ_Lj_TC1_rect_jj1_', 'Lj_variable': 'Lj1', 'Cj_variable': 'Cj'}, 'rect_jj2': {'rect': 'JJ_rect_Lj_TC1_rect_jj2', 'line': 'JJ_Lj_TC1_rect_jj2_', 'Lj_variable': 'Lj2', 'Cj_variable': 'Cj'}}, 'dissipatives': {'dielectrics_bulk': ['main']}, 'cos_trunc': 8, 'fock_trunc': 7, 'sweep_variable': 'Lj'}


In [14]:
# ── Run EPR analysis ──────────────────────────────────────────────────────
# run_epr() integrates E-field over the JJ rectangle and computes
# the inductive participation ratio p_mj and participation sign s_mj.
print("Running EPR field integration...")
eig_qb.run_epr()

# ── Access pyEPR DistributedAnalysis results ──────────────────────────────
# In this version of Qiskit-Metal, EPR data is stored on the renderer's
# epr_distributed_analysis object, NOT on eig_qb.epr_results.
da = eig_qb.sim.renderer.epr_distributed_analysis   # pyEPR DistributedAnalysis

print()
print("── EPR Participation (p_mj) ──")
print(da.results.p_mj)
print()
print("── EPR Sign (s_mj / PSR) ──")
print(da.results.sm_mj)


Running EPR field integration...
Design "Qbit_hfss_hfss" info:
	# eigenmodes    1
	# variations    1
Design "Qbit_hfss_hfss" info:
	# eigenmodes    1
	# variations    1

        energy_elec_all       = 1.04911864523588e-23
        energy_elec_substrate = 9.55072174437597e-24
        EPR of substrate = 91.0%

        energy_mag    = 4.39700156830788e-26
        energy_mag % of energy_elec_all  = 0.4%
        

Variation 0  [1/1]

  Mode 0 at 3.78 GHz   [1/1]
    Calculating ℰ_magnetic,ℰ_electric
       (ℰ_E-ℰ_H)/ℰ_E       ℰ_E       ℰ_H
               99.6%  5.246e-24 2.199e-26

    Calculating junction energy participation ration (EPR)
	method=`line_voltage`. First estimates:
	junction        EPR p_0j   sign s_0j    (p_capacitive)
		Energy fraction (Lj over Lj&Cj)= 97.04%
	rect_jj1        0.513053  (+)        0.0156249
		Energy fraction (Lj over Lj&Cj)= 97.04%
	rect_jj2        0.513336  (+)        0.0156335
		(U_tot_cap-U_tot_ind)/mean=0.03%
Calculating Qdielectric_main for mode 0 (0/0)

WARNING 12:00AM [hfss_report_f_convergence]: hfss_report_f_convergence: Could not set line width (cosmetic only, continuing): (-2147352567, 'Exception occurred.', (0, None, None, None, 0, -2147024382), None)
WARNING 12:00AM [__init__]: <p>Error: <class 'IndexError'></p>



ANALYSIS DONE. Data saved to:

C:\data-pyEPR\Project30\Qbit_hfss_hfss\2026-03-29 00-00-48.npz


	 Differences in variations:



 . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 
Variation 0

Starting the diagonalization
Finished the diagonalization
Pm_norm=
modes
0    1.00066
dtype: float64

Pm_norm idx =
   rect_jj1  rect_jj2
0      True      True
*** P (participation matrix, not normlz.)
   rect_jj1  rect_jj2
0  0.497502  0.497776

*** S (sign-bit matrix)
   s_rect_jj1  s_rect_jj2
0           1          -1
*** P (participation matrix, normalized.)
       0.5      0.5

*** Chi matrix O1 PT (MHz)
    Diag is anharmonicity, off diag is full cross-Kerr.
       146

*** Chi matrix ND (MHz) 
    160+0j

*** Frequencies O1 PT (MHz)
0    3633.362543
dtype: float64

*** Frequencies ND (MHz)
0    3627.157554+   0.000000j
dtype: complex128

*** Q_coupling
Empty DataFrame
Columns: []
Index: [0]


KeyError: '_Lj'

In [15]:
#eig_qb.sim.close()

---
## Step 7 — Qubit Frequency and Anharmonicity

From first-order EPR perturbation theory:

| Quantity | Formula |
|---|---|
| Corrected f_01 | `f_q = f_HFSS * (1 - p_total / 2)` |
| Anharmonicity | `alpha = - p_total^2 * Ec` |
| Ec estimate | invert `f_01 = sqrt(8*Ej_eff*Ec) - Ec` |
| Ej_eff | `= 2 * (Phi0/2pi)^2 / Lj`  (two JJs in parallel) |

The precise Ec and dressed frequency are calculated in Step 9.


In [16]:
h_planck = 6.626e-34
Phi0     = 2.068e-15

# ── HFSS bare eigenfrequency ──────────────────────────────────────────────
f_hfss_GHz = float(eig_qb.get_frequencies().iloc[0, 0])

# ── Inductive EPR for both junctions ─────────────────────────────────────
da   = eig_qb.sim.renderer.epr_distributed_analysis
p_mj = da.results.p_mj    # structure varies by pyEPR/Qiskit-Metal version

# ── Diagnostic: print the actual type and content ────────────────────────
print("DEBUG — p_mj structure:")
print(f"  type(da.results)  : {type(da.results)}")
print(f"  type(p_mj)        : {type(p_mj)}")
try:
    print(f"  p_mj.shape        : {p_mj.shape}")
    print(f"  p_mj.index        : {list(p_mj.index)}")
    print(f"  p_mj.columns      : {list(p_mj.columns)}")
except Exception:
    pass
print(f"  p_mj:\n{p_mj}")
print()

# ── deep_float: unwrap any nested dict/iterable until a float is found ────
def deep_float(v):
    """Recursively extract the first numeric value from nested dicts/iterables."""
    if isinstance(v, (int, float)):
        return float(v)
    try:
        return float(v)
    except (TypeError, ValueError):
        pass
    if hasattr(v, 'values'):
        return deep_float(next(iter(v.values())))
    if hasattr(v, '__iter__') and not isinstance(v, str):
        return deep_float(next(iter(v)))
    raise TypeError(f"Cannot extract float from {type(v)}: {v!r}")

# ── Extract participation for each junction using multiple fallback strategies
import pandas as pd

def get_junction_participation(p_mj, jj_key, mode=0):
    """
    Extract the inductive EPR participation for junction `jj_key`, mode `mode`.
    Handles the various structures that pyEPR / Qiskit-Metal may produce.
    """
    try:
        v = p_mj.iloc[mode][jj_key]
        return deep_float(v)
    except Exception:
        pass
    try:
        v = p_mj.loc[jj_key].iloc[mode]
        return deep_float(v)
    except Exception:
        pass
    try:
        inner = p_mj.iloc[0]
        if isinstance(inner, pd.DataFrame):
            v = inner.iloc[mode][jj_key]
            return deep_float(v)
        if isinstance(inner, pd.Series):
            v = inner[jj_key]
            return deep_float(v)
    except Exception:
        pass
    try:
        var0_results = da.results[list(da.results.keys())[0]]
        inner_pmj    = var0_results['p_mj'] if hasattr(var0_results, '__getitem__') \
                           else var0_results.p_mj
        v = inner_pmj.iloc[mode][jj_key] if isinstance(inner_pmj, pd.DataFrame) \
            else inner_pmj[jj_key]
        return deep_float(v)
    except Exception:
        pass
    raise RuntimeError(
        f"Could not extract p_mj for junction '{jj_key}'. "
        f"Check the DEBUG output above and update the extraction strategy."
    )

try:
    p_jj1 = get_junction_participation(p_mj, 'rect_jj1', mode=0)
    print(f"  Extracted p_jj1 = {p_jj1:.4f}")
except RuntimeError as e:
    print(f"  ERROR (jj1): {e}")
    p_jj1 = 0.0

try:
    p_jj2 = get_junction_participation(p_mj, 'rect_jj2', mode=0)
    print(f"  Extracted p_jj2 = {p_jj2:.4f}")
except RuntimeError as e:
    print(f"  ERROR (jj2): {e}")
    p_jj2 = 0.0

p_total = p_jj1 + p_jj2   # dual JJ: total inductive participation

print("=" * 58)
print(f"  HFSS bare eigenfrequency  : {f_hfss_GHz:.4f} GHz")
print(f"  p_inductive rect_jj1      : {p_jj1:.4f}")
print(f"  p_inductive rect_jj2      : {p_jj2:.4f}")
print(f"  p_total                   : {p_total:.4f}")
print("=" * 58)

# First-order EPR frequency correction
f_q_GHz = f_hfss_GHz * (1.0 - p_total / 2.0)
print(f"\n  Corrected f_01 (EPR O1)   : {f_q_GHz:.4f} GHz")

# ── Ej from Lj (dual JJ in parallel: Ej_eff = 2 * Ej_single) ────────────
Lj_nH      = float(eig_qb.sim.setup.vars.Lj.replace(' nH', '').strip())
Ej_GHz     = (Phi0 / (2 * np.pi))**2 / (Lj_nH * 1e-9) / (h_planck * 1e9)
Ej_eff_GHz = 2 * Ej_GHz   # two JJs in parallel: Ej_eff = Ej1 + Ej2

print(f"\n  Lj (each JJ)              : {Lj_nH:.1f} nH")
print(f"  Ej (single JJ)            : {Ej_GHz:.3f} GHz")
print(f"  Ej_eff (both JJs)         : {Ej_eff_GHz:.3f} GHz")

# ── Back-estimate Ec by inverting f01 ~ sqrt(8 Ej_eff Ec) - Ec ──────────
def f01_eq(Ec, Ej, target):
    return np.sqrt(8 * Ej * Ec) - Ec - target

try:
    Ec_GHz     = brentq(f01_eq, 0.01, 1.0, args=(Ej_eff_GHz, f_hfss_GHz))
    alpha_GHz  = -(p_total**2) * Ec_GHz
    EjEc_ratio = Ej_eff_GHz / Ec_GHz

    print(f"\n  Estimated Ec              : {Ec_GHz*1e3:.1f} MHz")
    print(f"  Estimated anharmonicity   : {alpha_GHz*1e3:.1f} MHz")
    print(f"  Estimated Ej_eff/Ec       : {EjEc_ratio:.1f}")

    ok_f  = "PASS" if 3.0 <= f_hfss_GHz <= 4.8 else "FAIL -- adjust Lj"
    ok_ej = "PASS" if EjEc_ratio > 50            else "FAIL -- reduce Lj"
    print(f"\n  Target: f_01 in 3-4.8 GHz : {f_hfss_GHz:.4f} GHz  [{ok_f}]")
    print(f"  Target: Ej_eff/Ec > 50    : {EjEc_ratio:.1f}       [{ok_ej}]")
except Exception as ex:
    print(f"  Could not estimate Ec: {ex}")
    Ec_GHz     = 0.140
    EjEc_ratio = Ej_eff_GHz / Ec_GHz


Design "Qbit_hfss_hfss" info:
	# eigenmodes    1
	# variations    1
Design "Qbit_hfss_hfss" info:
	# eigenmodes    1
	# variations    1
DEBUG — p_mj structure:
  type(da.results)  : <class 'addict.addict.Dict'>
  type(p_mj)        : <class 'addict.addict.Dict'>
  p_mj.shape        : {}
  p_mj.index        : []
  p_mj.columns      : []
  p_mj:
{}

  ERROR (jj1): Could not extract p_mj for junction 'rect_jj1'. Check the DEBUG output above and update the extraction strategy.
  ERROR (jj2): Could not extract p_mj for junction 'rect_jj2'. Check the DEBUG output above and update the extraction strategy.
  HFSS bare eigenfrequency  : 3.7796 GHz
  p_inductive rect_jj1      : 0.0000
  p_inductive rect_jj2      : 0.0000
  p_total                   : 0.0000

  Corrected f_01 (EPR O1)   : 3.7796 GHz


TypeError: 'Dict' object is not callable

---
## Step 8 — Substrate EPR and Surface Participation Ratio (SPR)

### 8a  Bulk substrate dielectric participation
The fraction of the total electric energy stored inside the bulk silicon
substrate.  Computed automatically during `run_epr()` and printed as
`EPR of substrate = XX.X%`.

**T1 budget (dielectric)**:
```
1/T1_diel = p_sub * omega_01 * tan(delta_sub)
```

### 8b  Surface Participation Ratio (SPR)
Energy density at nm-scale interfacial loss layers:

| Interface | Notation | t (nm) | eps_r | tan_delta |
|---|---|---|---|---|
| Metal–Air | MA | 3 | 10 | ~1e-3 |
| Metal–Substrate | MS | 2 | 11.4 | ~1e-3 |
| Substrate–Air | SA | 2 | 11.4 | ~1e-3 |

```
p_surf = (eps_r * eps_0 * t) / (2 * U_E) * Integral|E_parallel|^2 dA
```
Requires 2D Sheet objects in HFSS representing each interface.


In [17]:
# ── 8a: Bulk substrate EPR ────────────────────────────────────────────────
print("=== 8a: Bulk Substrate EPR ===")
print()

# EPRanalysis stores the electric energies as attributes after run_epr():
#   eig_qb.energy_elec      — total electric energy in the simulation volume [J]
#   eig_qb.energy_elec_sub  — electric energy inside the substrate dielectric [J]
# Substrate participation = energy_elec_sub / energy_elec
try:
    p_sub = float(eig_qb.energy_elec_sub) / float(eig_qb.energy_elec)
    print(f"  p_substrate (bulk Si)     : {p_sub:.4f}  ({p_sub*100:.1f}%)")
    print(f"  (energy_elec_sub / energy_elec)")
except (AttributeError, ZeroDivisionError, TypeError) as e:
    # Fallback: pyEPR also prints 'EPR of substrate = XX%' in the run_epr() output.
    # If the attribute isn't available, read the printed value above.
    p_sub = 0.92   # typical planar transmon on Si (replace with printed value)
    print(f"  [energy_elec_sub not available ({e})]")
    print(f"  Using fallback estimate    : p_sub ~ {p_sub}")
    print(f"  (check the run_epr() output above for 'EPR of substrate')")

# ── Dielectric-limited T1 ─────────────────────────────────────────────────
tan_delta_Si = 3e-6          # bulk Si at mK (high-purity crystalline)
omega_q      = 2 * np.pi * f_hfss_GHz * 1e9
Q_diel       = 1.0 / (p_sub * tan_delta_Si)
T1_diel_us   = Q_diel / omega_q * 1e6

print(f"\n  tan(delta) Si assumed     : {tan_delta_Si:.0e}")
print(f"  Dielectric Q              : {Q_diel:.2e}")
print(f"  Dielectric-limited T1     : {T1_diel_us:.0f} us")


=== 8a: Bulk Substrate EPR ===

  p_substrate (bulk Si)     : 0.9104  (91.0%)
  (energy_elec_sub / energy_elec)

  tan(delta) Si assumed     : 3e-06
  Dielectric Q              : 3.66e+05
  Dielectric-limited T1     : 15 us


In [18]:
# ── 8b: Surface Participation Ratio (SPR) ────────────────────────────────
print("=== 8b: Surface Participation Ratio (SPR) ===")
print()

eps_0 = 8.854e-12   # F/m

# Interface parameters (from literature)
surfaces = {
    'MA': {'eps_r': 10.0,  't_m': 3e-9,  'tan_d': 1e-3,  'label': 'metal-air'},
    'MS': {'eps_r': 11.4,  't_m': 2e-9,  'tan_d': 1e-3,  'label': 'metal-substrate'},
    'SA': {'eps_r': 11.4,  't_m': 2e-9,  'tan_d': 1e-3,  'label': 'substrate-air'},
}

print("  Full SPR calculation requires 2D Sheet objects in HFSS.")
print("  Create sheets at each interface, then use the HFSS field calculator.")
print()
print("  Template (pyEPR + HFSS COM interface):")
print("""
    renderer = eig_qb.sim.renderer
    pinfo    = renderer.pinfo
    U_E      = eig_qb.energy_elec     # total electric energy [J]  (fixed: not epr_results)

    sheet_names = {
        'MA': 'MA_surface',   # HFSS Sheet object name for metal-air interface
        'MS': 'MS_surface',   # HFSS Sheet object name for metal-substrate
        'SA': 'SA_surface',   # HFSS Sheet object name for substrate-air
    }

    for iface, sp in surfaces.items():
        # Field integral via HFSS calculator:
        # E_sq = pinfo.get_surface_tangential_Esquared(sheet_names[iface])
        # p_surf = sp['eps_r'] * eps_0 * sp['t_m'] * E_sq / (2 * U_E)
        # Q_surf = 1.0 / (p_surf * sp['tan_d'])
        # T1_surf_us = Q_surf / omega_q * 1e6
        pass
""")

# ── Proxy estimate (substrate EPR as proxy for MA) ────────────────────────
print("  Proxy estimate (order of magnitude):")
t_sub_m  = 500e-6    # 500 um Si wafer
p_MA_est = p_sub * (surfaces['MA']['eps_r'] * surfaces['MA']['t_m']) / \
                     (11.4 * t_sub_m)
T1_MA_us = (1.0 / (p_MA_est * surfaces['MA']['tan_d'])) / omega_q * 1e6

print(f"    p_MA (metal-air proxy)   : {p_MA_est:.2e}")
print(f"    T1_MA (tan_d=1e-3)       : {T1_MA_us:.0f} us")
print()
print("  Dominant loss: bulk substrate (p_sub ~ 90% for planar geometry).")
print("  Reduce MA surface loss by increasing pad_r or pad2pk_gap.")


=== 8b: Surface Participation Ratio (SPR) ===

  Full SPR calculation requires 2D Sheet objects in HFSS.
  Create sheets at each interface, then use the HFSS field calculator.

  Template (pyEPR + HFSS COM interface):

    renderer = eig_qb.sim.renderer
    pinfo    = renderer.pinfo
    U_E      = eig_qb.energy_elec     # total electric energy [J]  (fixed: not epr_results)

    sheet_names = {
        'MA': 'MA_surface',   # HFSS Sheet object name for metal-air interface
        'MS': 'MS_surface',   # HFSS Sheet object name for metal-substrate
        'SA': 'SA_surface',   # HFSS Sheet object name for substrate-air
    }

    for iface, sp in surfaces.items():
        # Field integral via HFSS calculator:
        # E_sq = pinfo.get_surface_tangential_Esquared(sheet_names[iface])
        # p_surf = sp['eps_r'] * eps_0 * sp['t_m'] * E_sq / (2 * U_E)
        # Q_surf = 1.0 / (p_surf * sp['tan_d'])
        # T1_surf_us = Q_surf / omega_q * 1e6
        pass

  Proxy estimate (order of magn

---
## Step 9 — Quantum Analysis: Ej, Ec, Ej/Ec and PSR

`analyze_all_variations()` reconstructs the full Josephson Hamiltonian
from the EPR participation ratios, using:
- Non-degenerate perturbation theory (ND): most accurate
- First-order perturbation (O1): faster cross-check

**Key outputs**:
| Symbol | Source in results dict | Units |
|---|---|---|
| f_01 | `results['f_ND'][0]` | GHz |
| Ec (= -alpha) | `-results['chi_ND'][0,0]` | GHz |
| alpha | `results['chi_ND'][0,0]` | GHz |
| Ej per JJ | `get_Ejs()` | GHz |
| Ej/Ec | `Ej_eff / Ec` | — |
| PSR (sign) | `epr_results['sign_mj']` | ±1 |


In [19]:
# ── Run full quantum analysis ─────────────────────────────────────────────
# FIX: analyze_all_variations() lives on pyEPR's QuantumAnalysis class, NOT
# on Qiskit-Metal's EPRanalysis wrapper (calling eig_qb.analyze_all_variations()
# raises AttributeError).  We instantiate QuantumAnalysis directly from the
# HDF5 file that run_epr() saved in Step 6, then call it on that object.
#
# SECOND FIX: da.data_filename may carry a stale timestamp — pyEPR records
# the time when run_epr() was *called*, but the .npz on disk is stamped when
# HFSS *finished* writing it (potentially minutes earlier).  Instead of
# trusting the filename directly, we scan the same directory for the real
# data file (the .npz that is NOT the HamiltonianResultsContainer).

import pyEPR as epr
import pathlib

da = eig_qb.sim.renderer.epr_distributed_analysis

# ── Locate the actual .npz data file on disk ─────────────────────────────
data_dir = pathlib.Path(da.data_filename).parent

npz_candidates = [
    f for f in data_dir.glob('*.npz')
    if 'HamiltonianResultsContainer' not in f.name
]

if not npz_candidates:
    raise FileNotFoundError(
        f"No pyEPR data .npz found in: {data_dir}\n"
        f"Expected by da.data_filename: {da.data_filename}"
    )

# Pick the most recently modified one (handles multiple runs in same folder)
actual_data_file = max(npz_candidates, key=lambda f: f.stat().st_mtime)

print(f"da.data_filename (may be stale) : {da.data_filename}")
print(f"Actual .npz found on disk       : {actual_data_file}")
print()

# ── Instantiate QuantumAnalysis from the real file ───────────────────────
qa = epr.QuantumAnalysis(str(actual_data_file))

# ── Run full quantum analysis ─────────────────────────────────────────────
print("Running analyze_all_variations()...")
print("  cos_trunc  = 8  — Taylor expansion of cos(phi) to 8th order")
print("  fock_trunc = 7  — Fock space size: levels |0> to |6>")
print()

qa.analyze_all_variations(
    cos_trunc  = 8,
    fock_trunc = 7,
)

print("Quantum analysis complete.")


WARNING 12:01AM [__init__]: <p>Error: <class 'IndexError'></p>


da.data_filename (may be stale) : C:\data-pyEPR\Project30\Qbit_hfss_hfss\2026-03-29 00-01-17.npz
Actual .npz found on disk       : C:\data-pyEPR\Project30\Qbit_hfss_hfss\2026-03-29 00-00-48.npz

	 Differences in variations:


Running analyze_all_variations()...
  cos_trunc  = 8  — Taylor expansion of cos(phi) to 8th order
  fock_trunc = 7  — Fock space size: levels |0> to |6>


 . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 
Variation 0

Starting the diagonalization
Finished the diagonalization
Pm_norm=
modes
0    1.00066
dtype: float64

Pm_norm idx =
   rect_jj1  rect_jj2
0      True      True
*** P (participation matrix, not normlz.)
   rect_jj1  rect_jj2
0  0.497502  0.497776

*** S (sign-bit matrix)
   s_rect_jj1  s_rect_jj2
0           1          -1
*** P (participation matrix, normalized.)
       0.5      0.5

*** Chi matrix O1 PT (MHz)
    Diag is anharmonicity, off diag is full cross-Kerr.
       146

*** Chi matrix ND (MHz) 
    160+0j

*** Fr

In [20]:
# ── Extract Ej, Ec, Ej/Ec ────────────────────────────────────────────────
variation = '0'

print("=" * 62)
print("  QUANTUM HAMILTONIAN PARAMETERS")
print("=" * 62)

# ── Josephson energies (dual JJ) ─────────────────────────────────────────
try:
    Ejs = qa.get_Ejs(variation=variation)
    print("\n  Josephson energy Ej per junction (GHz):")
    for jname, jval in Ejs.items():
        print(f"    {jname:20s}: {float(jval):.4f} GHz  ({float(jval)*1e3:.1f} MHz)")
    # Dual JJ in parallel: Ej_eff = sum of both (equal Lj -> Ej_eff = 2*Ej_single)
    Ej_eff_qa = sum(float(v) for v in Ejs.values())
    print(f"    {'Ej_eff (both JJs)':20s}: {Ej_eff_qa:.4f} GHz")
except Exception as ex:
    print(f"  [Ejs: {ex}]")
    Ej_eff_qa = Ej_eff_GHz

# ── Chi matrix -> Ec, anharmonicity ──────────────────────────────────────
try:
    res      = qa.results[variation]
    f_ND     = res['f_ND']
    f_O1     = res['f_O1']
    chi_ND   = res['chi_ND']

    Ec_ND    = -float(chi_ND[0, 0])
    alpha_ND =  float(chi_ND[0, 0])
    EjEc     = Ej_eff_qa / Ec_ND

    print(f"\n  Dressed qubit frequency:")
    print(f"    f_ND  (pert. theory)  : {float(f_ND[0]):.6f} GHz")
    print(f"    f_O1  (1st order)     : {float(f_O1[0]):.6f} GHz")

    print(f"\n  Charging energy  Ec   : {Ec_ND*1e3:.2f} MHz")
    print(f"  Anharmonicity  alpha  : {alpha_ND*1e3:.2f} MHz")

    print(f"\n  --- Ej/Ec (dual JJ, parallel) ---")
    print(f"    Ej_eff (both JJs)   : {Ej_eff_qa:.4f} GHz = {Ej_eff_qa*1e3:.1f} MHz")
    print(f"    Ec                  : {Ec_ND*1e3:.2f} MHz")
    print(f"    Ej_eff/Ec           : {EjEc:.1f}")

    ok_f  = "PASS" if 3.0 <= float(f_ND[0]) <= 4.8 else "FAIL"
    ok_ec = "PASS" if 120  <= Ec_ND*1e3    <= 160   else "FAIL"
    ok_ej = "PASS" if EjEc > 50                     else "FAIL"

    print(f"\n  --- Target check ---")
    print(f"    f_01 in 3-4.8 GHz   : {float(f_ND[0]):.4f} GHz  [{ok_f}]")
    print(f"    Ec  in 120-160 MHz  : {Ec_ND*1e3:.1f} MHz  [{ok_ec}]")
    print(f"    Ej_eff/Ec > 50      : {EjEc:.1f}         [{ok_ej}]")

    if ok_f == "FAIL":
        delta = float(f_ND[0]) - 3.8
        print(f"    --> {'Increase' if delta < 0 else 'Decrease'} Lj to "
              f"{'raise' if delta < 0 else 'lower'} f_01")
    if ok_ec == "FAIL":
        print("    --> Adjust pad_r or pad2pk_gap to change Ec")

except Exception as ex:
    print(f"  [chi_ND not available: {ex}]")


  QUANTUM HAMILTONIAN PARAMETERS

  Josephson energy Ej per junction (GHz):
    rect_jj1            : 6.0541 GHz  (6054.1 MHz)
    rect_jj2            : 6.0541 GHz  (6054.1 MHz)
  [Ejs: 'numpy.ndarray' object is not callable]


NameError: name 'Ej_eff_GHz' is not defined

In [21]:
# ── PSR: Participation Sign Ratio ────────────────────────────────────────
print("=" * 62)
print("  PARTICIPATION SIGN RATIO (PSR  /  s_mj)")
print("=" * 62)
print()

try:
    da    = eig_qb.sim.renderer.epr_distributed_analysis
    sm_mj = da.results.sm_mj

    print("DEBUG - sm_mj structure:")
    print(f"  type(sm_mj) : {type(sm_mj)}")
    print(f"  sm_mj       :\n{sm_mj}")
    print()

    def deep_sign(v):
        """Unwrap nested structures to get a +/-1 int."""
        if isinstance(v, (int, float)):
            return int(v)
        try:
            return int(float(v))
        except (TypeError, ValueError):
            pass
        if hasattr(v, 'values'):
            return deep_sign(next(iter(v.values())))
        if hasattr(v, '__iter__') and not isinstance(v, str):
            return deep_sign(next(iter(v)))
        raise TypeError(f"Cannot extract sign from {type(v)}: {v!r}")

    import pandas as pd

    def get_sign(sm_mj, jj_key):
        cols = list(sm_mj.columns) if hasattr(sm_mj, 'columns') else []
        col_idx = cols.index(jj_key) if jj_key in cols else 0
        for _get in [
            lambda: sm_mj.iloc[0][jj_key],
            lambda: sm_mj.iloc[0, col_idx],
            lambda: sm_mj.loc[jj_key].iloc[0],
        ]:
            try:
                raw = _get()
                if raw is not None:
                    return deep_sign(raw)
            except Exception:
                pass
        return None

    for jj_key in ['rect_jj1', 'rect_jj2']:
        sign_val = get_sign(sm_mj, jj_key)
        if sign_val is None:
            print(f"  {jj_key}: could not extract sign — check run_epr() output above.")
        else:
            sign_str = "+1  OK" if sign_val > 0 else "-1  WARN (check JJ LineString direction in HFSS)"
            print(f"  {jj_key}  s_0j = {sign_str}")

except Exception as ex:
    print(f"  [sm_mj not available: {ex}]")
    print("  PSR is printed in the run_epr() output above.")

print("""
  Interpretation:
    s = +1 : JJ current flows in the positive sense — correct orientation.
    s = -1 : reversed polarity; check that the JJ LineString in
             make_jj() points outward (island -> ground plane).

  Expected for TC1 (CircmonG, jj_angle=0, jj_offset_angle=180):
    rect_jj1 (east)  s_0j = +1
    rect_jj2 (west)  s_0j = +1
""")


  PARTICIPATION SIGN RATIO (PSR  /  s_mj)

DEBUG - sm_mj structure:
  type(sm_mj) : <class 'addict.addict.Dict'>
  sm_mj       :
{}

  rect_jj1: could not extract sign — check run_epr() output above.
  rect_jj2: could not extract sign — check run_epr() output above.

  Interpretation:
    s = +1 : JJ current flows in the positive sense — correct orientation.
    s = -1 : reversed polarity; check that the JJ LineString in
             make_jj() points outward (island -> ground plane).

  Expected for TC1 (CircmonG, jj_angle=0, jj_offset_angle=180):
    rect_jj1 (east)  s_0j = +1
    rect_jj2 (west)  s_0j = +1



In [22]:
# ── Final design summary ──────────────────────────────────────────────────
print("=" * 65)
print("  DESIGN SUMMARY  --  CircmonG (Dual-JJ Circular Transmon)")
print("=" * 65)

print(f"\n  Geometry (CircmonG 'TC1')")
print(f"    pad_r           : {circmon1.options.pad_r}  (qubit island radius)")
print(f"    pad2pk_gap      : {circmon1.options.pad2pk_gap}  (circular slot width)")
print(f"    jj_width        : {circmon1.options.jj_width}")
print(f"    jj_angle        : {circmon1.options.jj_angle} deg")
print(f"    jj_offset_angle : {circmon1.options.jj_offset_angle} deg")

print(f"\n  HFSS simulation")
print(f"    Lj (each JJ)   : {eig_qb.sim.setup.vars.Lj}")
print(f"    Cj             : {eig_qb.sim.setup.vars.Cj}")
print(f"    Bare f         : {f_hfss_GHz:.4f} GHz")

try:
    print(f"\n  Quantum Hamiltonian")
    print(f"    f_01         : {float(f_ND[0]):.4f} GHz")
    print(f"    Ec           : {Ec_ND*1e3:.2f} MHz")
    print(f"    alpha        : {alpha_ND*1e3:.2f} MHz")
    print(f"    Ej_eff       : {Ej_eff_qa:.4f} GHz  (both JJs in parallel)")
    print(f"    Ej_eff/Ec    : {EjEc:.1f}")
    print(f"\n  Loss estimates")
    print(f"    p_sub    : {p_sub:.3f} ({p_sub*100:.1f}%)")
    print(f"    T1_diel  : {T1_diel_us:.0f} us")
    print(f"    p_MA est : {p_MA_est:.2e}")
    print(f"    T1_MA est: {T1_MA_us:.0f} us")
except NameError:
    print("    (Complete Steps 4-9 to populate)")
print("=" * 65)


  DESIGN SUMMARY  --  CircmonG (Dual-JJ Circular Transmon)

  Geometry (CircmonG 'TC1')
    pad_r           : 200um  (qubit island radius)
    pad2pk_gap      : 100um  (circular slot width)
    jj_width        : 20um
    jj_angle        : 0 deg
    jj_offset_angle : 180 deg

  HFSS simulation
    Lj (each JJ)   : {}
    Cj             : 2 fF
    Bare f         : 3.7796 GHz

  Quantum Hamiltonian
    (Complete Steps 4-9 to populate)


---
## Tuning Guide

| Goal | Action | Effect |
|---|---|---|
| ↑ f_01 | Decrease `Lj` | ↑ Ej → ↑ f |
| ↓ f_01 | Increase `Lj` | ↓ Ej → ↓ f |
| ↑ Ec (more anharmonic) | Decrease `Rdisk` | ↓ CΣ → ↑ Ec |
| ↓ Ec | Increase `Rdisk` or `Wslot` | ↑ CΣ → ↓ Ec |
| Reduce substrate loss | Decrease `W` (smaller ground plane area) | Less substrate E-field |
| Reduce MA surface loss | Increase `Rdisk` + `W` | E-field more dilute over larger area |
| Stronger isolation | Increase `W` | More ground plane around the qubit |

**Lj convention**: Each JJ is assigned inductance `Lj` in HFSS.
The two JJs in parallel give `Lj_eff = Lj/2` and `Ej_eff = 2 * (Phi0/2pi)^2 / Lj`.


In [ ]:
# ── Optional: close HFSS ──────────────────────────────────────────────────
import gc

# Release field overlay COM objects from plot_fields() calls
try:
    eig_qb.sim.clear_fields()
except Exception:
    pass

# Release setup/solution COM handles held by the distributed analysis
try:
    da = eig_qb.sim.renderer.epr_distributed_analysis
    del da
except Exception:
    pass

# Force Python GC to decrement all remaining COM reference counts
gc.collect()
gc.collect()
# Now safe to close
eig_qb.sim.close()
gui.main_window.close()
print("Notebook complete.")
print("Uncomment the two lines above to close HFSS and the Metal GUI.")
